# 実績シート作成

指定した稼働年月の Salesforce 稼働データから、インボイス対応の実績シートを作成します。

## 実行手順

1. 下の実行条件で対象日を選ぶ（使うのは年と月）
2. Ctrl + F9 で実行する
3. 出力の最後に表示されるスプレッドシートの URL を開く

## 実施内容

1. 元シートを Google ドライブの「BO実績シート」へコピーする。名前は `YYYYMM_実績シート（インボイス対応）`
2. 指定年月の稼働明細を、実績シートの 7 行目以降へ出力する
3. 指定日が親契約の最終契約終了日より後なら、その親契約の稼働は出力しない。終了日と同じ日は残す
4. 3〜6 行目に Mstar / CW / TD / YT と、対象月の翌月末を支払い期日として入れる
5. 「口座マスタ」シートへ口座情報を出力する


In [ ]:
# @title 実行条件
# @markdown 対象日の **年** と **月** を稼働年月にします。日は使いません。スイッチをオフにすると、実行した日の年月になります。

USE_CUSTOM_DATE = True  # @param {type:"boolean"}
OUTPUTDATE = "2026-09-01"  # @param {"type":"date"}

# 必要ライブラリのインストール
!pip -q install gspread google-auth google-api-python-client "pyjwt[crypto]" requests

from google.colab import auth, drive, userdata
auth.authenticate_user()  # Googleアカウントで認可

import re
from decimal import Decimal, ROUND_HALF_UP
import datetime as dt
import json
import requests
import gspread
import google.auth
import time
import calendar
import jwt
from googleapiclient.discovery import build

"""
このスクリプトが行うこと（非エンジニア向けの概要）

1) Googleドライブ上の既存スプレッドシート（元シート）を、指定フォルダに対象月名でコピーします。
2) コピー先のスプレッドシートを開き、Salesforceオブジェクト（kado__c）の明細を出力します。
   - 1行目には「項目のAPI名」をヘッダとして出力します。
   - データ本体は7行目から（B7〜）書き込みます。
   - 指定日が親契約（keiyaku__c）の最終契約終了日より後の稼働は出力しません。
3) Salesforceの稼働オブジェクト（対象月分）を「流入元」別（Mster / CW / TD / YT）に集計し、
   - 3〜6行目の F列に Mstar / CW / TD / YT を出力します。
   - AR列（3〜6行目）に支払い期日を出力します。
   - AO列（3〜6行目）に AW列の文言と完全一致する行の AX列（支援費用）合計を出力します。
   - L列および O列（合計金額）の明細行への直接書き込みは行いません。

前提となる設定
- Google側: ColabでGoogle認証済みであること
- Salesforce側: JWT認証用シークレット（SF_CLIENT_ID / SF_USERNAME / SF_PRIVATE_KEY）を登録済みであること
"""

drive.mount('./gdrive')

# CSVファイルの出力先設定
drive_root_dir="./gdrive/My Drive/BO実績シート/"

# ===== 設定 =====
SOURCE_SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1_yfnpn5lMvw1nFxTP8TVYnyrSwdrxl73HALkdv_B-a4/edit"
TARGET_SHEET_TITLE = "実績シート"  # 処理対象シート名


def resolve_output_period(use_custom_date: bool, outputdate) -> tuple[int, int]:
    """出力対象の年月を返す。USE_CUSTOM_DATE=False のときは実行日の年月。"""
    if use_custom_date:
        target = dt.datetime.strptime(str(outputdate)[:10], "%Y-%m-%d").date()
        return target.year, target.month
    today = dt.date.today()
    return today.year, today.month


def soql_month_filter(field_name: str, year: int, month: int) -> str:
    """指定年月の日付項目に対する SOQL 条件を返す。"""
    first_day = dt.date(year, month, 1)
    last_day = calendar.monthrange(year, month)[1]
    last_date = dt.date(year, month, last_day)
    return f"{field_name} >= {first_day.isoformat()} AND {field_name} <= {last_date.isoformat()}"


def next_month_end(year: int, month: int) -> str:
    """指定年月の翌月末日を 'YYYY/MM/DD' 形式で返す。"""
    if month == 12:
        next_year, next_month = year + 1, 1
    else:
        next_year, next_month = year, month + 1
    last_day = calendar.monthrange(next_year, next_month)[1]
    return dt.date(next_year, next_month, last_day).strftime("%Y/%m/%d")


OUTPUT_YEAR, OUTPUT_MONTH = resolve_output_period(USE_CUSTOM_DATE, OUTPUTDATE)
if USE_CUSTOM_DATE:
    KADONENGETSU_FILTER = soql_month_filter("kadonengetsu_hiduke__c", OUTPUT_YEAR, OUTPUT_MONTH)
else:
    KADONENGETSU_FILTER = "kadonengetsu_hiduke__c = THIS_MONTH"
COPY_TITLE = f"{OUTPUT_YEAR:04d}{OUTPUT_MONTH:02d}_実績シート（インボイス対応）"  # コピー名（重複時は後で連番付与）
# =================

# スプレッドシートID抽出（URLでもIDでも可）
m = re.search(r"/d/([a-zA-Z0-9-_]+)", SOURCE_SPREADSHEET_URL)
source_id = m.group(1) if m else SOURCE_SPREADSHEET_URL

# 認証スコープ（DriveのコピーとSheets編集を許可）
scopes = [
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/spreadsheets",
]
creds, _ = google.auth.default(scopes=scopes)

def ensure_drive_folder_by_path(service, path_str: str) -> str:
    """My Drive配下のパスからフォルダを辿ってIDを返す（無ければ作成）。"""
    # パス正規化とセグメント抽出
    norm = path_str.replace('\\', '/')
    segments = [s for s in norm.split('/') if s not in ('', '.', 'gdrive')]
    # 'My Drive'より後ろを対象にする
    if 'My Drive' in segments:
        idx = segments.index('My Drive')
        segments = segments[idx + 1:]
    elif 'MyDrive' in segments:
        idx = segments.index('MyDrive')
        segments = segments[idx + 1:]
    # 親はルート（My Drive）
    parent_id = 'root'
    for name in segments:
        # 既存検索
        q = (
            f"mimeType = 'application/vnd.google-apps.folder' and "
            f"name = '{name.replace('\'', "\\'")}' and "
            f"'{parent_id}' in parents and trashed = false"
        )
        res = service.files().list(q=q, fields="files(id,name)", spaces='drive').execute()
        files = res.get('files', [])
        if files:
            parent_id = files[0]['id']
            continue
        # 無ければ作成
        meta = {
            'name': name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_id],
        }
        created = service.files().create(body=meta, fields='id').execute()
        parent_id = created['id']
    return parent_id

# Drive APIでファイルをコピー（指定フォルダ配下に作成）
drive_service = build("drive", "v3", credentials=creds)
target_folder_id = ensure_drive_folder_by_path(drive_service, drive_root_dir)

def get_available_name(service, parent_id: str, base_name: str) -> str:
    """同一フォルダ内で重複しない名前を返す。重複時は ' (2)', ' (3)'... を付与。"""
    # まずそのままの存在を確認
    esc = base_name.replace("'", "\\'")
    q = f"name = '{esc}' and '{parent_id}' in parents and trashed = false"
    res = service.files().list(q=q, fields="files(id)", spaces='drive').execute()
    if not res.get('files'):
        return base_name

    # 連番で空きを探す
    suffix = 2
    while True:
        candidate = f"{base_name} ({suffix})"
        esc2 = candidate.replace("'", "\\'")
        q2 = f"name = '{esc2}' and '{parent_id}' in parents and trashed = false"
        res2 = service.files().list(q=q2, fields="files(id)", spaces='drive').execute()
        if not res2.get('files'):
            return candidate
        suffix += 1

final_copy_name = get_available_name(drive_service, target_folder_id, COPY_TITLE)
copied = drive_service.files().copy(
    fileId=source_id,
    body={"name": final_copy_name, "parents": [target_folder_id]},
).execute()
new_id = copied["id"]
new_url = f"https://docs.google.com/spreadsheets/d/{new_id}/edit"

# 作成したスプレッドシートを指定ユーザーに編集権限で共有
SHARE_EMAILS = [
    "t-shigehiko@rosca.jp",
    "t-suzuki@rosca.jp",
    "k-satou@rosca.jp",
    "m-kuroda@rosca.jp",
]
for email in SHARE_EMAILS:
    try:
        drive_service.permissions().create(
            fileId=new_id,
            body={
                "type": "user",
                "role": "writer",
                "emailAddress": email,
            },
            fields="id",
            sendNotificationEmail=False,
        ).execute()
    except Exception:
        # 共有に失敗しても他処理は継続
        pass

# gspreadでコピー先を開く（コピー直後の伝播遅延に備えてリトライ）
gc = gspread.authorize(creds)

max_attempts = 8
for attempt in range(max_attempts):
    try:
        sh = gc.open_by_key(new_id)
        break
    except gspread.exceptions.APIError:
        if attempt == max_attempts - 1:
            raise
        time.sleep(min(60, 2 ** attempt))

for attempt in range(max_attempts):
    try:
        ws = sh.worksheet(TARGET_SHEET_TITLE)
        break
    except gspread.exceptions.APIError:
        if attempt == max_attempts - 1:
            raise
        time.sleep(min(60, 2 ** attempt))

print(new_url)

# ===== Salesforce 認証（JWT Bearer フロー） =====
# Colab のシークレットに登録済みの値を取得
SF_CLIENT_ID = userdata.get('SF_CLIENT_ID')
SF_USERNAME = userdata.get('SF_USERNAME')
SF_PRIVATE_KEY = userdata.get('SF_PRIVATE_KEY')

if not SF_CLIENT_ID or not SF_USERNAME or not SF_PRIVATE_KEY:
    raise RuntimeError('Salesforce シークレット (SF_CLIENT_ID / SF_USERNAME / SF_PRIVATE_KEY) が見つかりません。')

# 一般的なSalesforceのトークンエンドポイント
SF_TOKEN_URL = 'https://login.salesforce.com/services/oauth2/token'


def _normalize_private_key(raw_key: str) -> str:
    """
    Colab Secret に保存した秘密鍵を PEM 形式に正規化する。
    \n エスケープを実改行へ変換して利用する。
    """
    return str(raw_key).strip().replace("\\n", "\n")


def sf_auth():
    now = int(time.time())
    claims = {
        'iss': SF_CLIENT_ID,
        'sub': SF_USERNAME,
        'aud': 'https://login.salesforce.com',
        'exp': now + 180,
    }
    assertion = jwt.encode(claims, _normalize_private_key(SF_PRIVATE_KEY), algorithm='RS256')
    token_resp = requests.post(
        SF_TOKEN_URL,
        data={
            'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
            'assertion': assertion,
        },
        timeout=60,
    )
    token_resp.raise_for_status()
    token_json = token_resp.json()
    return token_json['access_token'], token_json['instance_url']


access_token, instance_url = sf_auth()

# ===== Salesforce から kado__c を直接取得 =====
headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json',
}

# 取得対象項目（API名）
headers_sf = [
    'Id',
    'keiyakukaishibi__c',
    'keiyakusyuryobi__c',
    'shitashinkiorsaihan__c',
    'gizyutusyamei__c',
    'gizyutusyamei_furigana__c',
    'uekigyomei__c',
    'uekigyobunrui__c',
    'uetanto__c',
    'uekingaku__c',
    'kozameigi__c',
    'keiyakukeitaikubun__c',
    'shitatanto__c',
    'shitakingaku__c',
    'arari__c',
    'ueseikyu_teisyutukigen__c',
    'ueseikyu_genpon__c',
    'ueshiharaisaito_math__c',
    'ueshiharaisaitobi_math__c',
    'uetesuryo__c',
    'uesyukusaizitunobai__c',
    'shitashiharaisaito_math__c',
    'shitashiharaisaitobi_math__c',
    'shitatesuryo__c',
    'shitasyukusaizitunobai__c',
    'uekinmuhyoteisyutukigen_math__c',
    'uekinmuhyoteisyutuzikan__c',
    'zyucyubi__c',
    'shitaseisan_kotei__c',
    'ryunyumoto__c',
    'shitakinmuhyoteisyutukigen__c',
    'zissekitanto__c',
    'kinmuhyokaisyu__c',
    'koutuhinoumu__c',
    'uezissekikingaku_zeibetsu__c',
    'uezissekikingaku_zeikomi__c',
    'uekingakuwariai__c',
    'ueshiharaikizitsu__c',
    'seikyusyoumu__c',
    'shitazissekikingaku__c',
    'shitazissekikingaku_zeikomi__c',
    'shitakengakuwariai__c',
    'shitashiharaikizitsu__c',
    'shitaseikyusyozyuryo__c',
    'kazei_menzeizigyosya__c',
    'tekikakuseikyusyohakkou__c',
    'seikyusyosashikaehitsuyo__c',
    'ownership__c',
    'ryunyumototesuryo__c',
    'kingakukakunin_backoffice__c',
    'Wcheck__c',
    'zissekiarari__c',
]

# ownership__c / ryunyumototesuryo__c は必ず
# seikyusyosashikaehitsuyo__c の直後に配置する
def _move_fields_after_anchor(fields, anchor, targets):
    out = [f for f in fields if f not in targets]
    if anchor in out:
        insert_at = out.index(anchor) + 1
        for t in targets:
            if t in fields:
                out.insert(insert_at, t)
                insert_at += 1
    return out


headers_sf = _move_fields_after_anchor(
    headers_sf,
    'seikyusyosashikaehitsuyo__c',
    ['ownership__c', 'ryunyumototesuryo__c'],
)

# 出力開始位置設定（ヘッダは1行目、明細データは7行目から）
start_col = 2  # B列開始
start_row_data = 7  # データは7行目から

# 一部項目は Salesforce 値を使わず、シート上の数式で出力する
ue_tax_exclusive_field = 'uezissekikingaku_zeibetsu__c'
ue_tax_inclusive_field = 'uezissekikingaku_zeikomi__c'
shita_tax_exclusive_field = 'shitazissekikingaku__c'
shita_tax_inclusive_field = 'shitazissekikingaku_zeikomi__c'
ue_ratio_field = 'uekingakuwariai__c'
shita_ratio_field = 'shitakengakuwariai__c'

formula_fields = {
    ue_tax_inclusive_field,
    shita_tax_inclusive_field,
    ue_ratio_field,
    shita_ratio_field,
}
# 出力列と取得元項目が異なるもの（出力先: 取得元）
source_field_overrides = {
    'kazei_menzeizigyosya__c': 'zigyosyabunrui_text__c',
}

query_fields = [f for f in headers_sf if f not in formula_fields]
for src in source_field_overrides.values():
    if src not in query_fields:
        query_fields.append(src)
# 税込/税抜判定用の補助項目も取得
for tax_flag_field in ('uezeikominuki__c', 'shitazeikominuki__c'):
    if tax_flag_field not in query_fields:
        query_fields.append(tax_flag_field)
# YouTrust手数料は youtrusttesuryo__c をそのまま使うため取得対象に含める
if 'youtrusttesuryo__c' not in query_fields:
    query_fields.append('youtrusttesuryo__c')
# 親契約の最終契約終了日で除外判定するため、参照IDも取得する（シート列には出さない）
OYAKEIYAKU_ID_FIELD = "oyakeiyakuid__c"
PARENT_CONTRACT_OBJECT = "keiyaku__c"
PARENT_FINAL_END_LABEL = "最終契約終了日"
if OYAKEIYAKU_ID_FIELD not in query_fields:
    query_fields.append(OYAKEIYAKU_ID_FIELD)

ue_tax_exclusive_idx = headers_sf.index(ue_tax_exclusive_field)
ue_tax_inclusive_idx = headers_sf.index(ue_tax_inclusive_field)
shita_tax_exclusive_idx = headers_sf.index(shita_tax_exclusive_field)
shita_tax_inclusive_idx = headers_sf.index(shita_tax_inclusive_field)
ue_ratio_idx = headers_sf.index(ue_ratio_field)
shita_ratio_idx = headers_sf.index(shita_ratio_field)
ue_kingaku_idx = headers_sf.index('uekingaku__c')
shita_kingaku_idx = headers_sf.index('shitakingaku__c')


def fetch_all_soql_records(soql: str):
    records = []
    query_url = f"{instance_url}/services/data/v57.0/query?q={requests.utils.quote(soql)}"
    while True:
        resp = requests.get(query_url, headers=headers, timeout=120)
        if not resp.ok:
            try:
                print("SOQL request failed:", resp.status_code, resp.text)
            except Exception:
                pass
        resp.raise_for_status()
        payload = resp.json()
        records.extend(payload.get('records', []) or [])
        if payload.get('done'):
            break
        next_url = payload.get('nextRecordsUrl')
        if not next_url:
            break
        query_url = f"{instance_url}{next_url}"
    return records


def fetch_object_field_names(object_api_name: str):
    url = f"{instance_url}/services/data/v57.0/sobjects/{object_api_name}/describe"
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    payload = resp.json() or {}
    return {f.get('name') for f in (payload.get('fields') or []) if f.get('name')}


def _convert_to_tax_exclusive_if_needed(amount_raw, tax_type_raw):
    """
    税区分が「税込」の場合のみ、金額を税抜(÷1.1)へ変換する。
    変換不能な値は元の値を返す。
    """
    tax_type = str(tax_type_raw or '').strip()
    if tax_type != '税込':
        return amount_raw
    if amount_raw in (None, ''):
        return amount_raw
    text = str(amount_raw).strip().replace(',', '')
    try:
        value = float(text)
    except Exception:
        return amount_raw
    converted = value / 1.1
    # 整数金額は四捨五入で整数化、少数は小数第2位まで
    if abs(converted - round(converted)) < 1e-9:
        return int(round(converted))
    return round(converted, 2)


# UT手数料と YT手数料は YOUTRUST手数料と同じ。表示と合計条件を揃える
OWNERSHIP_LABEL_ALIASES = {
    'UT手数料': 'YOUTRUST手数料',
    'YT手数料': 'YOUTRUST手数料',
}
YOUTRUST_FEE_LABELS = ('YOUTRUST手数料', 'YT手数料')
NO_FEE_OWNERSHIP_LABEL = '手数料なし'
HOURLY_WAGE_LABEL = '時給'
HOURLY_MONTHLY_HOURS = 160
SUPPORT_FEE_RATE = Decimal('0.1')


def _ownership_label(value):
    """支援種別の表示名。空はそのまま、UT手数料と YT手数料は YOUTRUST手数料にする。"""
    text = str(value or '').strip()
    if not text:
        return value
    return OWNERSHIP_LABEL_ALIASES.get(text, text)


def _youtrust_fee_value(rec, fallback_raw):
    """
    支援種別が手数料なしのときは 0 を返す。
    YouTrust のときは youtrusttesuryo__c をそのまま返す。
    最低額補正は Salesforce 側で済みのため、ここでは加工しない。
    """
    ownership = _ownership_label(rec.get('ownership__c'))
    ownership = str(ownership or '').strip()
    if ownership == NO_FEE_OWNERSHIP_LABEL:
        return 0
    if ownership not in YOUTRUST_FEE_LABELS:
        return fallback_raw
    return rec.get('youtrusttesuryo__c')


def _is_hourly_wage(rec) -> bool:
    return str(rec.get('shitaseisan_kotei__c') or '').strip() == HOURLY_WAGE_LABEL


def _to_number(value):
    if isinstance(value, bool) or value in (None, ''):
        return None
    if isinstance(value, (int, float)):
        return Decimal(str(value))
    text = str(value).strip().replace(',', '').replace('¥', '').replace('￥', '')
    if not text:
        return None
    try:
        return Decimal(text)
    except Exception:
        return None


def _round_money(value: Decimal):
    """金額は円未満を四捨五入し、整数円にする。"""
    return int(value.quantize(Decimal('1'), rounding=ROUND_HALF_UP))


def _hourly_monthly_amount(amount_raw, tax_type_raw):
    """時給単価を税抜にしたあと、160時間をかけて月額にする。"""
    exclusive = _convert_to_tax_exclusive_if_needed(amount_raw, tax_type_raw)
    number = _to_number(exclusive)
    if number is None:
        return exclusive
    return _round_money(number * HOURLY_MONTHLY_HOURS)


def _support_fee_cell(rec, fallback_raw, sheet_row):
    """
    支援費用。下金額の10%になる行は、そのセルを参照する数式にする。
    下金額を後から直すと支援費も追従する。
    手数料なしは0。YouTrustと支援種別が空の行は Salesforce の値のまま。
    """
    ownership = _ownership_label(rec.get('ownership__c'))
    ownership = str(ownership or '').strip()
    shita_cell = gspread.utils.rowcol_to_a1(sheet_row, start_col + shita_kingaku_idx)
    if ownership == NO_FEE_OWNERSHIP_LABEL:
        return f"=IF({shita_cell}=\"\",\"\",0)"
    if ownership in YOUTRUST_FEE_LABELS or not ownership:
        return _youtrust_fee_value(rec, fallback_raw)
    rate = format(SUPPORT_FEE_RATE, 'f')
    return f"=IF({shita_cell}=\"\",\"\",ROUND({shita_cell}*{rate},0))"


def _col_letter(col_num: int) -> str:
    """1始まりの列番号から列記号（A, B, ...）を返す"""
    a1 = gspread.utils.rowcol_to_a1(1, col_num)
    return ''.join(ch for ch in a1 if not ch.isdigit())


def _strip_yen_prefix(value):
    """通貨記号（￥/¥）を除いた表示用文字列を返す"""
    if value in (None, ''):
        return value
    return str(value).replace('￥', '').replace('¥', '').strip()


soql_detail = (
    "SELECT "
    + ",".join(query_fields)
    + f" FROM kado__c WHERE {KADONENGETSU_FILTER}"
)
available_kado_fields = fetch_object_field_names('kado__c')
query_fields_available = [f for f in query_fields if f in available_kado_fields]
missing_query_fields = [f for f in query_fields if f not in available_kado_fields]
if missing_query_fields:
    print("WARN: kado__c に存在しない項目をSELECTから除外:", ", ".join(missing_query_fields))
if not query_fields_available:
    raise RuntimeError("kado__c の取得対象項目が1つも有効ではありません。項目API名を確認してください。")

soql_detail = (
    "SELECT "
    + ",".join(query_fields_available)
    + f" FROM kado__c WHERE {KADONENGETSU_FILTER}"
)
kado_records = fetch_all_soql_records(soql_detail)


def _parse_sf_date(value):
    if value in (None, ""):
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        return dt.datetime.strptime(text[:10], "%Y-%m-%d").date()
    except ValueError:
        return None


def _escape_soql_literal(value: str) -> str:
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def _sf_id_lookup_keys(value):
    text = str(value or "").strip()
    if not text:
        return []
    keys = [text]
    if len(text) >= 15:
        keys.append(text[:15])
    return list(dict.fromkeys(keys))


def _specified_output_date() -> dt.date:
    """比較に使う指定日。USE_CUSTOM_DATE=False のときは実行日。"""
    if USE_CUSTOM_DATE:
        return dt.datetime.strptime(str(OUTPUTDATE)[:10], "%Y-%m-%d").date()
    return dt.date.today()


def _parent_final_end_field_name() -> str:
    url = f"{instance_url}/services/data/v57.0/sobjects/{PARENT_CONTRACT_OBJECT}/describe"
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    fields = (resp.json() or {}).get("fields") or []
    matches = [
        f.get("name")
        for f in fields
        if str(f.get("label") or "").strip() == PARENT_FINAL_END_LABEL and f.get("name")
    ]
    if len(matches) == 1:
        return matches[0]
    labels = sorted({
        str(f.get("label") or "")
        for f in fields
        if "終了" in str(f.get("label") or "")
    })
    hint = "、".join(label for label in labels if label) or "(終了を含む項目なし)"
    raise RuntimeError(
        f"{PARENT_CONTRACT_OBJECT} に項目ラベル「{PARENT_FINAL_END_LABEL}」が見つかりません。"
        f" 終了を含む項目: {hint}"
    )


def _fetch_parent_final_end_dates(parent_ids, end_field: str) -> dict:
    unique_ids = []
    for parent_id in parent_ids:
        text = str(parent_id or "").strip()
        if text and text not in unique_ids:
            unique_ids.append(text)
    mapping = {}
    chunk_size = 100
    for start in range(0, len(unique_ids), chunk_size):
        chunk = unique_ids[start:start + chunk_size]
        id_list = ",".join(f"'{_escape_soql_literal(item)}'" for item in chunk)
        soql = (
            f"SELECT Id, {end_field} FROM {PARENT_CONTRACT_OBJECT} "
            f"WHERE Id IN ({id_list})"
        )
        for rec in fetch_all_soql_records(soql):
            end_date = _parse_sf_date(rec.get(end_field))
            for key in _sf_id_lookup_keys(rec.get("Id")):
                mapping[key] = end_date
    return mapping


def _exclude_kado_after_parent_end(records):
    """指定日が親契約の最終契約終了日より後なら、その親契約の稼働を除く。"""
    if OYAKEIYAKU_ID_FIELD not in available_kado_fields:
        raise RuntimeError(
            f"kado__c に {OYAKEIYAKU_ID_FIELD} がないため、親契約の終了日では除外できません。"
        )
    specified = _specified_output_date()
    end_field = _parent_final_end_field_name()
    end_dates = _fetch_parent_final_end_dates(
        [rec.get(OYAKEIYAKU_ID_FIELD) for rec in records],
        end_field,
    )
    kept = []
    skipped = 0
    unresolved = 0
    for rec in records:
        parent_id = str(rec.get(OYAKEIYAKU_ID_FIELD) or "").strip()
        end_date = None
        found = False
        for key in _sf_id_lookup_keys(parent_id):
            if key in end_dates:
                end_date = end_dates[key]
                found = True
                break
        if parent_id and found and end_date is not None and specified > end_date:
            skipped += 1
            continue
        if parent_id and not found:
            unresolved += 1
        kept.append(rec)
    print(
        f"親契約の{PARENT_FINAL_END_LABEL}（{end_field}）より後の稼働を {skipped} 件スキップ"
        f"（指定日 {specified.isoformat()}、出力 {len(kept)} 件）"
    )
    if unresolved:
        print(f"WARN: 親契約が見つからない稼働 {unresolved} 件はそのまま出力します。")
    return kept


kado_records = _exclude_kado_after_parent_end(kado_records)

# 2次元配列を作成（ヘッダ順に値を配置）
data_values = []
for row_idx, rec in enumerate(kado_records):
    sheet_row = start_row_data + row_idx
    row_vals = []
    for col_idx, col in enumerate(headers_sf):
        if col == ue_tax_inclusive_field:
            # 同行の税別金額セルを参照する式（例: =AF7*1.1）を出力
            base_cell_a1 = gspread.utils.rowcol_to_a1(sheet_row, start_col + ue_tax_exclusive_idx)
            row_vals.append(f"=IFERROR({base_cell_a1}*1.1,\"\")")
        elif col == shita_tax_inclusive_field:
            base_cell_a1 = gspread.utils.rowcol_to_a1(sheet_row, start_col + shita_tax_exclusive_idx)
            row_vals.append(f"=IFERROR({base_cell_a1}*1.1,\"\")")
        elif col == ue_ratio_field:
            ue_tax_excl_cell = gspread.utils.rowcol_to_a1(sheet_row, start_col + ue_tax_exclusive_idx)
            ue_kingaku_cell = gspread.utils.rowcol_to_a1(sheet_row, start_col + ue_kingaku_idx)
            row_vals.append(f"=IFERROR({ue_tax_excl_cell}/{ue_kingaku_cell},\"\")")
        elif col == shita_ratio_field:
            shita_tax_excl_cell = gspread.utils.rowcol_to_a1(sheet_row, start_col + shita_tax_exclusive_idx)
            shita_kingaku_cell = gspread.utils.rowcol_to_a1(sheet_row, start_col + shita_kingaku_idx)
            row_vals.append(f"=IFERROR({shita_tax_excl_cell}/{shita_kingaku_cell},\"\")")
        else:
            source_col = source_field_overrides.get(col, col)
            v = rec.get(source_col)
            if col == 'uekingaku__c':
                v = _convert_to_tax_exclusive_if_needed(v, rec.get('uezeikominuki__c'))
                if _is_hourly_wage(rec):
                    number = _to_number(v)
                    if number is not None:
                        v = _round_money(number * HOURLY_MONTHLY_HOURS)
            elif col == 'shitakingaku__c':
                v = _convert_to_tax_exclusive_if_needed(v, rec.get('shitazeikominuki__c'))
                if _is_hourly_wage(rec):
                    number = _to_number(v)
                    if number is not None:
                        v = _round_money(number * HOURLY_MONTHLY_HOURS)
            elif col == 'ownership__c':
                # AW列: UT手数料と YT手数料は YOUTRUST手数料として表示し、AO6の合計に含める
                v = _ownership_label(v)
            elif col == 'ryunyumototesuryo__c':
                # AX列: 下金額の10%は数式。手数料なしは0。YouTrustは値のまま
                v = _support_fee_cell(rec, v, sheet_row)
            elif col == 'shitashiharaisaitobi_math__c':
                v = _strip_yen_prefix(v)
            if isinstance(v, bool):
                row_vals.append(v)
            elif v is None:
                row_vals.append("")
            else:
                row_vals.append(str(v))
    data_values.append(row_vals)

# 行・列が足りなければ拡張（対象全列数でカウント）
num_cols_all = len(headers_sf)
required_rows = start_row_data + max(len(data_values), 1) - 1
if ws.row_count < required_rows:
    ws.add_rows(required_rows - ws.row_count)

required_cols = start_col + max(num_cols_all, 1) - 1
if ws.col_count < required_cols:
    ws.add_cols(required_cols - ws.col_count)

# ヘッダ（1行目、B列以降）: すべてのAPI名を出力

if headers_sf:
    # 旧ヘッダ残骸を避けるため、ヘッダ行は一度クリアしてから再出力
    ws.batch_clear([f"{gspread.utils.rowcol_to_a1(1, start_col)}:ZZ1"])
    header_range = (
        gspread.utils.rowcol_to_a1(1, start_col)
        + ":"
        + gspread.utils.rowcol_to_a1(1, start_col + num_cols_all - 1)
    )
    ws.update(values=[headers_sf], range_name=header_range, value_input_option='USER_ENTERED')
    print(
        "header index:",
        "seikyusyosashikaehitsuyo__c=",
        headers_sf.index('seikyusyosashikaehitsuyo__c'),
        "ownership__c=",
        headers_sf.index('ownership__c'),
        "ryunyumototesuryo__c=",
        headers_sf.index('ryunyumototesuryo__c'),
    )

# データ（7行目以降、B列開始）:
# zissekiarari__c はテンプレート側の数式を維持するため書き込み対象から除外
if data_values:
    preserve_formula_fields = {'zissekiarari__c'}
    write_indices = [i for i, h in enumerate(headers_sf) if h not in preserve_formula_fields]

    def iter_consecutive_segments(indices):
        if not indices:
            return
        seg_start = indices[0]
        prev = indices[0]
        for idx in indices[1:]:
            if idx != prev + 1:
                yield seg_start, prev
                seg_start = idx
            prev = idx
        yield seg_start, prev

    for seg_start, seg_end in iter_consecutive_segments(write_indices):
        data_block = [[row[i] for i in range(seg_start, seg_end + 1)] for row in data_values]
        data_range = (
            gspread.utils.rowcol_to_a1(start_row_data, start_col + seg_start)
            + ":"
            + gspread.utils.rowcol_to_a1(start_row_data + len(data_values) - 1, start_col + seg_end)
        )
        ws.update(values=data_block, range_name=data_range, value_input_option='USER_ENTERED')

    # X列（shitashiharaisaitobi_math__c）: テンプレートの通貨書式による￥表示を抑止
    x_col = start_col + headers_sf.index('shitashiharaisaitobi_math__c')
    x_letter = _col_letter(x_col)
    last_data_row = start_row_data + len(data_values) - 1
    ws.format(
        f"{x_letter}{start_row_data}:{x_letter}{last_data_row}",
        {"numberFormat": {"type": "NUMBER", "pattern": "#,##0"}},
    )
else:
    print(f'kado__c に {OUTPUT_YEAR:04d}/{OUTPUT_MONTH:02d} 条件の明細データが見つかりませんでした。')

# ===== 稼働オブジェクト（対象月）の流入元別合計と支払い期日の出力 =====
# 稼働オブジェクトのAPI名（環境により異なる可能性あり）
# Colab のシークレット 'Salesforce_kado_object' があればそれを優先（未設定時は 'kado__c'）
try:
    KADO_OBJECT_API = userdata.get('Salesforce_kado_object')
except Exception:
    KADO_OBJECT_API = None
if not KADO_OBJECT_API:
    KADO_OBJECT_API = 'kado__c'

# SOQLで対象月分、流入元が Mster/CW/TD/YT の手数料合計を流入元別に集計
soql = (
    "SELECT ryunyumoto__c, SUM(ryunyumototesuryo__c) totalFee "
    f"FROM {KADO_OBJECT_API} "
    f"WHERE {KADONENGETSU_FILTER} AND ryunyumoto__c IN ('Mster','CW','TD','YT') "
    "GROUP BY ryunyumoto__c"
)

query_url = f"{instance_url}/services/data/v57.0/query?q={requests.utils.quote(soql)}"
resp = requests.get(query_url, headers=headers, timeout=60)
if resp.status_code == 200:
    res_json = resp.json()
    records = res_json.get('records', [])
    # 期待するキーを0で初期化
    totals = {"Mster": 0, "CW": 0, "TD": 0, "YT": 0}
    for rec in records:
        source = rec.get('ryunyumoto__c')
        amount = rec.get('totalFee') or rec.get('expr0') or 0
        if source in totals:
            totals[source] = amount

    # O列（15列目）・AR列（44列目）・AX列（50列目）まで存在することを保証、行も6行目まで保証
    REQUIRED_MAX_COL = 50
    if ws.col_count < REQUIRED_MAX_COL:
        ws.add_cols(REQUIRED_MAX_COL - ws.col_count)
    if ws.row_count < 6:
        ws.add_rows(6 - ws.row_count)

    # SOQLの 'Mster' を表示上の 'Mstar' にマッピング
    totals_display = {"Mstar": totals.get("Mster", 0), "CW": totals.get("CW", 0), "TD": totals.get("TD", 0), "YT": totals.get("YT", 0)}

    # 翌月末の日付を計算（出力対象月ベース）
    pay_due_str = next_month_end(OUTPUT_YEAR, OUTPUT_MONTH)

    # AW列の文言と完全一致する指定列（AX列・支援費用）を合計（明細は7行目以降）
    def support_fee_sum_formula(label: str) -> str:
        escaped = label.replace('"', '""')
        return f'=SUMIFS($AX$7:$AX,$AW$7:$AW,"{escaped}")'

    support_fee_by_row = {
        3: "Mstar手数料",
        4: "CW手数料",
        5: "TD手数料",
        6: "YOUTRUST手数料",
    }

    # AO列（3〜6行目）: AX列（支援費用）の合計数式
    ao_values = [
        [support_fee_sum_formula(support_fee_by_row[row])]
        for row in range(3, 7)
    ]
    ws.update(values=ao_values, range_name="AO3:AO6", value_input_option='USER_ENTERED')

    # AT列（3〜6行目）: テンプレート残りの数式を削除
    ws.batch_clear(["AT3:AT6"])

    # 3行目: Mstar + 支払い期日
    ws.update(values=[["Mstar"]], range_name="F3:F3", value_input_option='USER_ENTERED')
    ws.update(values=[[pay_due_str]], range_name="AR3:AR3", value_input_option='USER_ENTERED')

    # 4行目: CW + 支払い期日
    ws.update(values=[["CW"]], range_name="F4:F4", value_input_option='USER_ENTERED')
    ws.update(values=[[pay_due_str]], range_name="AR4:AR4", value_input_option='USER_ENTERED')

    # 5行目: TD + 支払い期日
    ws.update(values=[["TD"]], range_name="F5:F5", value_input_option='USER_ENTERED')
    ws.update(values=[[pay_due_str]], range_name="AR5:AR5", value_input_option='USER_ENTERED')

    # 6行目: YT + 支払い期日
    ws.update(values=[["YT"]], range_name="F6:F6", value_input_option='USER_ENTERED')
    ws.update(values=[[pay_due_str]], range_name="AR6:AR6", value_input_option='USER_ENTERED')
else:
    # 失敗時は詳細を出力
    try:
        print('SOQLエラー:', resp.status_code, resp.text)
    except Exception:
        pass

# ===== 口座マスタシートに bankaccount__c 明細を出力 =====
BANKACCOUNT_FIELDS = [
    'Id',
    'Name',
    'bankcode__c',
    'branchcode__c',
    'accounttype__c',
    'accountnumber__c',
    'accountholder__c',
]

soql_bankaccount = "SELECT " + ",".join(BANKACCOUNT_FIELDS) + " FROM bankaccount__c"
bankaccount_records = fetch_all_soql_records(soql_bankaccount)

# シートを取得（存在しない場合は作成）
try:
    ws_master = sh.worksheet("口座マスタ")
except Exception:
    ws_master = sh.add_worksheet(title="口座マスタ", rows=100, cols=50)

# 既存データをすべてクリア
ws_master.clear()

# 列メタ（API名）と明細行を作成（A1起点で出力）
headers_sf2 = list(BANKACCOUNT_FIELDS)
num_cols2 = len(headers_sf2)
data_values2 = []
for rec in bankaccount_records:
    row_vals = []
    for col in headers_sf2:
        v = rec.get(col)
        if isinstance(v, bool):
            row_vals.append(v)
        elif v is None:
            row_vals.append("")
        else:
            row_vals.append(str(v))
    data_values2.append(row_vals)

# 行・列が足りなければ拡張（ヘッダは1行目A列、データは2行目A列から）
start_col_master = 1  # A列
start_row_master_data = 2  # データは2行目から

required_rows2 = start_row_master_data + max(len(data_values2), 1) - 1
if ws_master.row_count < required_rows2:
    ws_master.add_rows(required_rows2 - ws_master.row_count)

required_cols2 = start_col_master + max(num_cols2, 1) - 1
if ws_master.col_count < required_cols2:
    ws_master.add_cols(required_cols2 - ws_master.col_count)

# ヘッダ（1行目、A列以降）
if headers_sf2:
    header_range2 = (
        gspread.utils.rowcol_to_a1(1, start_col_master)
        + ":"
        + gspread.utils.rowcol_to_a1(1, start_col_master + num_cols2 - 1)
    )
    ws_master.update(values=[headers_sf2], range_name=header_range2, value_input_option='USER_ENTERED')

# データ（2行目以降、A列開始）
if data_values2:
    data_range2 = (
        gspread.utils.rowcol_to_a1(start_row_master_data, start_col_master)
        + ":"
        + gspread.utils.rowcol_to_a1(start_row_master_data + len(data_values2) - 1, start_col_master + num_cols2 - 1)
    )
    ws_master.update(values=data_values2, range_name=data_range2, value_input_option='USER_ENTERED')
else:
    print('bankaccount__c に明細データが見つかりませんでした。')

pass
